# Versión comentada (línea por línea)

En esta versión, cada celda de código incluye comentarios **numerados** que explican **qué estamos haciendo en cada línea**.
La intención es didáctica: que puedas leer el código como si fuera un procedimiento paso a paso.


# Notebook 3 — Modelo pinhole: del mundo 3D a la imagen 2D

Del PDF: componente geométrico y ecuación de proyección perspectiva:
$$x/X = f/Z, \quad y/Y = f/Z$$

Ejemplo cotidiano: **¿por qué un coche lejano se ve pequeño?**
Ejemplo industrial: **medición dimensional con backlight (siluetas)**.


In [ ]:
# (1) Importamos librerías/módulos que vamos a usar.
import numpy as np
# (2) Importamos librerías/módulos que vamos a usar.
import matplotlib.pyplot as plt

## A) Proyectar puntos 3D a 2D (pinhole)
Haremos un “mundo” simple: un cubo (o puntos) y lo proyectamos.


In [ ]:
# 1) Definimos una función de proyección pinhole.
# (1) Definimos la función `proyectar_pinhole`.
def proyectar_pinhole(P, f):
    # (2) Ejecutamos esta instrucción como parte del procedimiento.
    """Proyecta puntos 3D P (N x 3) a 2D usando x = f*X/Z, y = f*Y/Z."""
    # (3) Definimos/asignamos la variable `X`.
    X = P[:, 0]
    # (4) Definimos/asignamos la variable `Y`.
    Y = P[:, 1]
    # (5) Definimos/asignamos la variable `Z`.
    Z = P[:, 2]
    # Evitamos división entre cero.
    # (6) Definimos/asignamos la variable `eps`.
    eps = 1e-9
    # (7) Definimos/asignamos la variable `x`.
    x = f * X / (Z + eps)
    # (8) Definimos/asignamos la variable `y`.
    y = f * Y / (Z + eps)
    # (9) Regresamos el resultado de la función.
    return np.column_stack([x, y])

# 2) Creamos puntos 3D de un cuadrado a diferentes profundidades.
# (10) Definimos/asignamos la variable `cuadrado`.
cuadrado = np.array([
    # (11) Ejecutamos esta instrucción como parte del procedimiento.
    [-1, -1, 0],
    # (12) Ejecutamos esta instrucción como parte del procedimiento.
    [ 1, -1, 0],
    # (13) Ejecutamos esta instrucción como parte del procedimiento.
    [ 1,  1, 0],
    # (14) Ejecutamos esta instrucción como parte del procedimiento.
    [-1,  1, 0]
# (15) Ejecutamos esta instrucción como parte del procedimiento.
], dtype=float)

# 3) Duplicamos ese cuadrado a dos profundidades (cerca y lejos).
# (16) Definimos/asignamos la variable `P_cerca`.
P_cerca = cuadrado + np.array([0, 0, 3])   # Z=3
# (17) Definimos/asignamos la variable `P_lejos`.
P_lejos = cuadrado + np.array([0, 0, 8])   # Z=8

# (18) Definimos/asignamos la variable `f`.
f = 1.5
# (19) Definimos/asignamos la variable `p2_cerca`.
p2_cerca = proyectar_pinhole(P_cerca, f)
# (20) Definimos/asignamos la variable `p2_lejos`.
p2_lejos = proyectar_pinhole(P_lejos, f)

# 4) Dibujamos las proyecciones.
# (21) Creamos una nueva figura para graficar.
plt.figure(figsize=(6,6))
# (22) Ejecutamos esta instrucción como parte del procedimiento.
plt.plot(*p2_cerca.T, 'o-', label='Cuadrado cerca (Z=3)')
# (23) Ejecutamos esta instrucción como parte del procedimiento.
plt.plot(*p2_lejos.T, 'o-', label='Cuadrado lejos (Z=8)')
# (24) Ejecutamos esta instrucción como parte del procedimiento.
plt.axhline(0, linewidth=1)
# (25) Ejecutamos esta instrucción como parte del procedimiento.
plt.axvline(0, linewidth=1)
# (26) Ejecutamos esta instrucción como parte del procedimiento.
plt.gca().set_aspect('equal', 'box')
# (27) Agregamos un título a la gráfica.
plt.title('Proyección pinhole: el objeto lejano se ve más pequeño')
# (28) Ejecutamos esta instrucción como parte del procedimiento.
plt.legend()
# (29) Renderizamos las gráficas en pantalla.
plt.show()

## B) “Objeto grande lejos” vs “objeto pequeño cerca” (ambigüedad 2D)
El PDF dice que una imagen 2D es ambigua: **un objeto grande lejos puede verse igual a uno pequeño cerca**.
Simulémoslo: creamos dos cuadrados distintos que se proyectan casi igual.


In [ ]:
# 5) Cuadrado grande lejos
# (1) Definimos/asignamos la variable `P_grande_lejos`.
P_grande_lejos = (2.0 * cuadrado) + np.array([0,0,10])

# 6) Cuadrado pequeño cerca
# (2) Definimos/asignamos la variable `P_pequeno_cerca`.
P_pequeno_cerca = (1.0 * cuadrado) + np.array([0,0,5])

# (3) Definimos/asignamos la variable `p2_1`.
p2_1 = proyectar_pinhole(P_grande_lejos, f)
# (4) Definimos/asignamos la variable `p2_2`.
p2_2 = proyectar_pinhole(P_pequeno_cerca, f)

# (5) Creamos una nueva figura para graficar.
plt.figure(figsize=(6,6))
# (6) Ejecutamos esta instrucción como parte del procedimiento.
plt.plot(*p2_1.T, 'o-', label='Grande lejos (escala 2, Z=10)')
# (7) Ejecutamos esta instrucción como parte del procedimiento.
plt.plot(*p2_2.T, 'o--', label='Pequeño cerca (escala 1, Z=5)')
# (8) Ejecutamos esta instrucción como parte del procedimiento.
plt.gca().set_aspect('equal', 'box')
# (9) Agregamos un título a la gráfica.
plt.title('Ambigüedad: proyecciones muy parecidas')
# (10) Ejecutamos esta instrucción como parte del procedimiento.
plt.legend()
# (11) Renderizamos las gráficas en pantalla.
plt.show()

### Lectura intuitiva
En la imagen 2D, ambas figuras pueden ocupar un tamaño similar. Por eso visión es un **problema inverso**: queremos recuperar 3D a partir de 2D, pero falta información (profundidad).


## C) Micro-aplicación industrial: medición de ancho con backlight (silueta)
Con contraluz, convertimos el objeto en una silueta y medimos dimensiones en píxeles.
Aquí simularemos una silueta como rectángulo binario y mediremos su ancho.


In [ ]:
# 7) Creamos una imagen binaria simple: un rectángulo blanco sobre fondo negro.
# (1) Ejecutamos esta instrucción como parte del procedimiento.
H, W = 200, 300
# (2) Definimos/asignamos la variable `img`.
img = np.zeros((H, W), dtype=np.uint8)
# (3) Ejecutamos esta instrucción como parte del procedimiento.
img[70:140, 90:230] = 255  # objeto

# 8) Medición: ancho en píxeles de la silueta (columna mínima y máxima donde hay blanco).
# (4) Definimos/asignamos la variable `cols`.
cols = np.where(img.max(axis=0) > 0)[0]
# (5) Definimos/asignamos la variable `ancho_px`.
ancho_px = int(cols[-1] - cols[0] + 1)

# (6) Creamos una nueva figura para graficar.
plt.figure(figsize=(7,4))
# (7) Mostramos una matriz como imagen.
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
# (8) Agregamos un título a la gráfica.
plt.title(f'Silueta (ancho medido = {ancho_px} px)')
# (9) Configuramos/ocultamos ejes para visualizar mejor.
plt.axis('off')
# (10) Renderizamos las gráficas en pantalla.
plt.show()

### Nota
Para convertir píxeles a milímetros se calibra (con un patrón conocido) y se consideran intrínsecos/extrínsecos (matriz K, [R|t]) como menciona el PDF.
